In [ ]:
# Silver Validation Script
import os
import pandas as pd
from src.database import get_engine

# Load RDS credentials from environment variables
RDS_HOST = os.getenv("RDS_HOST")
RDS_PORT = int(os.getenv("RDS_PORT", "5432"))
RDS_DATABASE = os.getenv("RDS_DATABASE")
RDS_USER = os.getenv("RDS_USER")
RDS_PASSWORD = os.getenv("RDS_PASSWORD")

BRONZE_SCHEMA = os.getenv("BRONZE_SCHEMA", "bronze_group6")
SILVER_SCHEMA = os.getenv("SILVER_SCHEMA", "silver_group6")

# Create SQLAlchemy engine
engine = get_engine()

In [2]:
def count_columns(schema: str, table: str) -> int:
    query = f"""
    SELECT COUNT(*) AS column_count
    FROM information_schema.columns
    WHERE table_schema = '{schema}'
    AND table_name = '{table}'
    """
    
    df = pd.read_sql(query, engine)
    return df["column_count"].iloc[0]

In [4]:
tables = [
    ("products", "dim_products"),
    ("users", "dim_users"),
    ("orders", "fct_orders"),
    ("order_line_items", "fct_order_lines")
]

results = []

for bronze_table, silver_table in tables:
    bronze_cols = count_columns(BRONZE_SCHEMA, bronze_table)
    silver_cols = count_columns(SILVER_SCHEMA, silver_table)
    
    status =  "OK" if silver_cols < bronze_cols else "PROBLEM"
    
    results.append({
        "Table": bronze_table,
        "Bronze_cols": bronze_cols,
        "Silver_cols": silver_cols,
        "Status": status
    })

df_results = pd.DataFrame(results)
df_results

,Table,Bronze_cols,Silver_cols,Status
0,products,21,17,OK
1,users,28,20,OK
2,orders,31,25,OK
3,order_line_items,16,13,OK


In [5]:
def get_columns(schema, table):
    query = f"""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = '{schema}'
    AND table_name = '{table}'
    """
    return pd.read_sql(query, engine)["column_name"].tolist()


bronze_cols = set(get_columns(BRONZE_SCHEMA, "products"))
silver_cols = set(get_columns(SILVER_SCHEMA, "dim_products"))

removed = [col for col in bronze_cols if col.startswith("_")]

print(" Colonnes internes attendues supprimées :")
print(removed)

print("\n Toujours présentes dans Silver :")
print(set(removed) & silver_cols)

 Colonnes internes attendues supprimées :
['_warehouse_location', '_internal_cost_usd', '_internal_cost_code', '_supplier_id']

 Toujours présentes dans Silver :
set()


In [7]:
query = """
SELECT schemaname, COUNT(*) AS nb_tables
FROM pg_tables
WHERE schemaname IN ('bronze_group6', 'silver_group6', 'gold_group6')
GROUP BY schemaname
ORDER BY schemaname;
"""

df_tables = pd.read_sql(query, engine)
df_tables

,schemaname,nb_tables
0,bronze_group6,6
1,gold_group6,3
2,silver_group6,4
